# PROBES preprocessed images — browse & pick a source

This notebook shows the **preprocessed** PROBES galaxy images (the actual training data of
the score-based prior) so you can browse the sample, **pick one by index**, and then push it
through the SIE+shear lens and run the posterior sampler against the trained prior.

Preprocessing (see `data/preprocess.py`): g-band FITS → center-crop to 256×256 → clip to
`[0, 5.5]` → min-max normalise to `[-1, 1]` (`x_bar = 2*clip(x,0,5.5)/5.5 - 1`). Each `.npy`
is a single-channel `(1, 256, 256)` float32 array.

In [ ]:
import glob
import math
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

PROJECT_DIR = Path('/home/yd388/rds/hpc-work/DIS-Project-Lensed-Galaxy')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().parent  # notebook lives in notebooks/
DATA_DIR   = PROJECT_DIR / 'data' / 'gals_gband_norm'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'probes_final'

files = sorted(glob.glob(str(DATA_DIR / '*.npy')))
assert files, f'No .npy files found in {DATA_DIR}'
print(f'{len(files)} preprocessed PROBES images in {DATA_DIR}')

# Match src/sample.py: arrays stay in [-1, 1], while intensity plots invert the
# preprocessing scale back to clipped PROBES flux and use a log color scale.
FLUX_A = 5.5
FLUX_VMIN = 1e-2

def to_display_flux(img, floor=1e-3):
    if isinstance(img, torch.Tensor):
        return (FLUX_A * (img.detach().cpu() + 1.0) / 2.0).clamp(min=floor).numpy()
    return np.clip(FLUX_A * (np.asarray(img) + 1.0) / 2.0, floor, None)

flux_kw = dict(cmap='magma', norm=LogNorm(vmin=FLUX_VMIN, vmax=FLUX_A), origin='lower')


In [2]:
def load_image(idx):
    """Load one preprocessed PROBES image -> (256, 256) float32 in [-1, 1]."""
    arr = np.load(files[idx]).astype(np.float32)
    if arr.ndim == 3:
        arr = arr[0]
    return arr

def name_of(idx):
    return Path(files[idx]).stem

## Gallery

Each tile is titled with its **global index** and source name. Change `start` to page through
the full sample (e.g. `show_gallery(48)`, `show_gallery(96)`, ...).

In [ ]:
def show_gallery(start=0, n=48, ncols=8, cmap='magma', display='log_flux'):
    """Display preprocessed PROBES images starting at index `start`."""
    n = min(n, len(files) - start)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 2.15 * nrows))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel():
        ax.axis('off')
    for j in range(n):
        idx = start + j
        ax = axes.ravel()[j]
        img = load_image(idx)
        if display == 'linear':
            ax.imshow(img, cmap=cmap, origin='lower', vmin=-1, vmax=1)
        else:
            ax.imshow(to_display_flux(img), **flux_kw)
        ax.set_title(f'{idx}: {name_of(idx)}', fontsize=6)
    plt.tight_layout()
    plt.show()

show_gallery(0, 48)


In [8]:
def find(substr):
    """List (index, name) for every image whose name contains `substr` (case-insensitive)."""
    hits = [(i, name_of(i)) for i in range(len(files)) if substr.lower() in name_of(i).lower()]
    for i, nm in hits:
        print(f'{i:5d}  {nm}')
    return hits

# e.g. find('NGC')

## Pick one

Set `PICK` to the index of the galaxy you want to use as the true source.

In [ ]:
PICK = 15  # <- index from the gallery above

src_img = load_image(PICK)
print(f'Picked [{PICK}] {name_of(PICK)}  shape={src_img.shape}  '
      f'range=[{src_img.min():.3f}, {src_img.max():.3f}]')

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(to_display_flux(src_img), **flux_kw)
ax.set_title(f'[{PICK}] {name_of(PICK)}')
ax.axis('off')
fig.colorbar(im, fraction=0.046, pad=0.04, label='Intensity')
plt.show()


## Load the trained prior

Same checkpoint and loading convention as `full_sample.ipynb` (custom `train_prior.py` EMA weights).

In [10]:
from score_models import ScoreModel, NCSNpp

ckpt = torch.load(OUTPUT_DIR / 'latest.pt', map_location='cpu', weights_only=False)
hp = ckpt['score_model_hyperparameters']

net = NCSNpp(**hp)
net.load_state_dict(ckpt['ema_model'])
model = ScoreModel(model=net, sigma_min=hp['sigma_min'], sigma_max=hp['sigma_max'], device=device)
model = model.to(device)
print(f"Loaded EMA at step {ckpt['step']}, sigma_max={model.sde.sigma_max}")

KeyboardInterrupt: 

## Lens the picked image and sample the posterior

Build the caustics SIE + external-shear forward model, lens the picked PROBES galaxy, add
observation noise, then reconstruct it with the convolved-likelihood posterior sampler
(Adam et al. 2022) using the trained prior.

In [ ]:
import caustics

IMAGE_SIZE = 256
PIXELSCALE = 0.04
Z_LENS, Z_SOURCE = 0.5, 1.5

cosmology = caustics.FlatLambdaCDM(name='cosmo')
lens = caustics.SIE(
    cosmology=cosmology, name='lens',
    z_l=Z_LENS, z_s=Z_SOURCE,
    x0=0.0, y0=0.0,
    q=0.7, phi=math.pi / 6, Rein=1.2,
)
source = caustics.Pixelated(
    name='source', x0=0.0, y0=0.0,
    pixelscale=PIXELSCALE, shape=(IMAGE_SIZE, IMAGE_SIZE),
)
sim = caustics.LensSource(
    lens=lens, source=source,
    pixelscale=PIXELSCALE, pixels_x=IMAGE_SIZE, pixels_y=IMAGE_SIZE,
    name='sim',
)
sim.to(device)


def lens_forward(sim, x):
    """Match src/sample.py: out-of-FOV rays map to -1 empty sky, not 0 mid-flux."""
    return sim({'source': {'image': x + 1.0}}) - 1.0


@torch.no_grad()
def posterior_sample(model, sim, y, sigma_y, steps=2000, n_samples=4):
    """Convolved-likelihood posterior sampler (Adam et al. 2022, eqs. 19, 20)."""
    device = next(model.parameters()).device
    H = W = y.shape[-1]
    sde = model.sde

    t_grid = torch.linspace(1.0, 1e-3, steps + 1, device=device)
    x = sde.prior([n_samples, 1, H, W]).sample().to(device)

    for i in range(steps):
        t_scalar = t_grid[i]
        t_batch = t_scalar.expand(n_samples)
        h = (t_grid[i] - t_grid[i + 1])
        var = sigma_y ** 2 + sde.sigma(t_scalar) ** 2
        g = sde.diffusion(t_batch, x)

        with torch.enable_grad():
            x_req = x.detach().requires_grad_(True)
            preds = torch.stack([lens_forward(sim, x_req[b, 0])
                                 for b in range(n_samples)])
            log_lik = -0.5 * ((y - preds) ** 2).sum() / var
            grad_ll = torch.autograd.grad(log_lik, x_req)[0]

        score_post = model.score(t_batch, x) + grad_ll
        z = torch.randn_like(x)
        x = x + (g ** 2) * score_post * h + g * z * h.sqrt()

    return x


In [ ]:
# Use the picked PROBES galaxy as the true source
src = torch.from_numpy(src_img).to(device)

with torch.no_grad():
    lensed = lens_forward(sim, src)

NOISE_SIGMA = 0.02
obs = lensed + NOISE_SIGMA * torch.randn_like(lensed)

post = posterior_sample(model, sim, obs, sigma_y=NOISE_SIGMA,
                        steps=2000, n_samples=4).cpu()

fig, axes = plt.subplots(1, 6, figsize=(18, 3.2))
axes[0].imshow(to_display_flux(src), **flux_kw)
axes[0].set_title(f'True source
[{PICK}] {name_of(PICK)}')
axes[1].imshow(to_display_flux(obs), **flux_kw)
axes[1].set_title('Observation (lensed + noise)')
for k in range(4):
    axes[2 + k].imshow(to_display_flux(post[k, 0]), **flux_kw)
    axes[2 + k].set_title(f'Posterior {k}')
for a in axes:
    a.axis('off')
plt.tight_layout()
plt.show()
